# Bahth بحث — image search in Arabic, Turkish and English

## Part 1: how well do existing models already do?

Before fine-tuning anything, this notebook measures three pretrained image-text models on the Flickr8k test images, with the same queries in three languages. Each caption is a search query; a query counts as a hit at K if its own photo is among the top K of 1,000.

Code: https://github.com/readaloush/bahth

**Settings:** Internet on, GPU accelerator.

In [ ]:
!pip -q install open_clip_torch sentence-transformers
!git clone -q https://github.com/readaloush/bahth.git
%cd bahth
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Data

Flickr8k photos with the original English captions, the Arabic captions of ElJundi et al. (2020) and the native Turkish captions of TasvirEt (Unal et al., 2016).

In [ ]:
from bahth.data import prepare, captions, splits
root = prepare('data'); caps = captions(root); test = splits(root)['test']
for lang in caps:
    print(lang, len(caps[lang]), 'images with captions;', sum(len(caps[lang].get(n, [])) for n in test), 'test queries')
n = test[0]
for lang in caps: print(lang, caps[lang].get(n))

## Baselines

In [ ]:
!python -m bahth.baselines --data data

In [ ]:
import json, pandas as pd
r = json.load(open('results/baselines.json'))['models']
rows = [{'model': m, 'language': l, **{k: round(v * 100, 1) for k, v in s.items() if k.startswith('R@')}, 'median rank': s['median_rank']} for m, d in r.items() if 'error' not in d for l, s in d.items()]
pd.DataFrame(rows).set_index(['model', 'language'])

In [ ]:
!cp data/turkish.json results/turkish_matches.json 2>/dev/null; ls -la results